# Mecab 사용자 사전용 csv 파일 구성

In [9]:
import pandas as pd
# jamo 라이브러리가 없다면 pip install jamo 필요
# 만약 설치가 어렵다면, 아래 has_coda 함수 대신 단순히 'T'로 통일해도 동작은 합니다(권장은 아님).
import jamo 

def has_coda(word):
    """
    한글 단어의 마지막 글자에 받침(종성)이 있는지 확인하는 함수
    """
    if not word: return False
    last_char = word[-1]
    if 0xAC00 <= ord(last_char) <= 0xD7A3:
        return (ord(last_char) - 0xAC00) % 28 != 0
    return False

def create_mecab_user_dic_from_csv(input_csv_path, output_path="user_dic.csv"):
    """
    법령용어 CSV를 읽어 Mecab 사용자 사전으로 변환
    조건: 공백이 2개 이상 포함된 문장형 데이터는 제외
    """
    # 1. CSV 파일 읽기
    try:
        df = pd.read_csv(input_csv_path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(input_csv_path, encoding='cp949')

    print(f"총 원본 데이터 개수: {len(df)}개")

    user_dic_data = []
    
    # 데이터프레임 컬럼명 확인 ('법령용어명' 컬럼이 있어야 함)
    target_column = '법령용어명'
    if target_column not in df.columns:
        print(f"Error: CSV 내에 '{target_column}' 컬럼이 없습니다.")
        return

    # NaN 제거 및 문자열 변환
    terms = df[target_column].dropna().astype(str)
    
    excluded_count = 0
    
    for term in terms:
        clean_term = term.strip()
        
        # --- [수정된 필터링 로직] ---
        
        # 1. 공백 개수 확인
        # 공백이 2개 이상이면 '설명문/문장'으로 간주하고 제외
        if clean_term.count(' ') >= 2:
            excluded_count += 1
            continue
            
        # # 2. 너무 짧거나(1글자) 너무 긴 경우(30글자 이상) 안전장치 (선택 사항)
        # # 법률 용어 중 '소' 같은 1글자도 있지만, 오탐 가능성이 있어 보통 제외하거나 수동 검토함
        # if len(clean_term) < 2 or len(clean_term) > 30:
        #     excluded_count += 1
        #     continue
            
        # ---------------------------

        # 종성 유무 판단 (마지막 글자 기준)
        coda_flag = "T" if has_coda(clean_term) else "F"
        
        # Mecab 사전 포맷 생성
        # 품사는 NNP(고유명사)로 강제, 우선순위(비용)는 0
        line = [clean_term, 0, 0, 0, "NNP", "*", coda_flag, clean_term, "*", "*", "*", "*"]
        user_dic_data.append(line)
    
    # CSV 저장
    pd.DataFrame(user_dic_data).to_csv(output_path, index=False, header=False, encoding='utf-8')
    
    print("-" * 50)
    print(f"변환 완료: {output_path}")
    print(f" - 등록된 용어: {len(user_dic_data)}개")
    print(f" - 제외된 용어: {excluded_count}개 (공백 2개 이상 또는 길이 부적절)")
    print("-" * 50)

# 실행 예시 (가지고 계신 파일 경로를 입력하세요)
# create_mecab_user_dic_from_csv("법령용어목록.csv")

In [10]:
create_mecab_user_dic_from_csv("data\법령용어목록_전체.csv")

총 원본 데이터 개수: 29330개
--------------------------------------------------
변환 완료: user_dic.csv
 - 등록된 용어: 17840개
 - 제외된 용어: 11490개 (공백 2개 이상 또는 길이 부적절)
--------------------------------------------------


### Mecab Test

In [5]:
from konlpy.tag import Mecab
import re

# Mecab 인스턴스 생성 (사용자 사전을 컴파일해서 적용했다면 자동으로 반영됨)
# 만약 사용자 사전 경로를 별도로 지정해야 한다면 Mecab(dicpath="...") 사용
mecab = Mecab() 

def preprocess_text(text):
    """
    1. 텍스트 정제 (특수문자 등)
    2. 형태소 분석
    3. 품사 필터링 (조사, 어미, 접사 등 제거)
    """
    # 1. 기본적인 텍스트 정제 (한글만 남기고 싶다면 주석 해제)
    # text = re.sub(r"[^가-힣\s]", "", text) 
    
    # 2. 형태소 분석
    pos_tags = mecab.pos(text)
    
    filtered_tokens = []
    
    for word, pos in pos_tags:
        # 3. 품사 필터링 로직
        # 명사(N): 일반명사(NNG), 고유명사(NNP), 의존명사(NNB)
        if pos.startswith('N'):
            filtered_tokens.append(word)
            
        # 용언(V): 동사(VV), 형용사(VA) -> 어간(Stem)만 필요
        # Mecab은 기본적으로 원형 복원(Lemmatization)을 완벽하게 해주지 않음.
        # 활용형(예: '하였습니다')이 그대로 나오거나 분리됨.
        # 연구 목적상 '용언'은 노이즈가 많을 수 있어서, 
        # 1차적으로는 '명사' 위주로 불용어를 선정하는 것이 깔끔할 수 있음.
        # 그래도 포함한다면 아래와 같이 추가:
        elif pos.startswith('V'):
             # 어미가 붙어있는 경우 단순화 (이 부분은 추가적인 원형 복원 로직이 필요할 수 있음)
             filtered_tokens.append(word)
    
    return filtered_tokens